# MediaPulse — Phase 4: Exploratory Data Analysis (EDA)

**Project:** MediaPulse — Media Performance & Audience Analytics  
**Target Role:** Wipro Analyst L1 (Business Analyst / Data Analyst - Media)  
**Objective:** Uncover empirical media performance dynamics, engagement patterns, category variations, and channel retention on the cleaned YouTube India trending dataset (`cleaned_youtube_india.csv`).

---

## 7 Core Analytical Investigations:
1. **Trending Activity Over Time:** Daily snapshot counts vs unique active videos.
2. **Category Performance:** Volume, median views, and engagement rates by content genre.
3. **Channel Performance:** Top 10 channels by unique trending content and audience traction.
4. **Engagement Dynamics:** Views vs. likes and comments; log-log scatter and empirical distributions.
5. **Time to Observed Trending:** Days elapsed between upload and earliest trending appearance at unique-video grain.
6. **Trending Persistence:** Distribution of distinct observed trending snapshot dates per unique video.
7. **Publication Patterns:** Upload distributions across weekdays and UTC hours.

> **Analytical Rigor Notice:** Multi-day observations of the same video represent longitudinal persistence across discrete daily snapshots. Video-level KPIs are aggregated using `views.max()` to avoid double-counting cumulative view counts. Summed peak views represent cumulative observed video-view totals rather than unduplicated audience reach or impressions.

### 1. Environment Setup & Data Loading

We import data science and visualization libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) alongside our modular `src.eda` and `src.metrics` packages.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config, eda, metrics

# Load cleaned dataset
cleaned_csv_path = config.DATA_PROCESSED_DIR / 'cleaned_youtube_india.csv'
df = pd.read_csv(cleaned_csv_path)
df['trending_date'] = pd.to_datetime(df['trending_date'])
df['publish_time'] = pd.to_datetime(df['publish_time'])

print(f"Loaded Cleaned Dataset: {len(df):,} rows x {len(df.columns)} columns")
print(f"Unique Videos: {df['video_id'].nunique():,}")
print(f"Unique Channels: {df['channel_title'].nunique():,}")
print(f"Unique Categories: {df['category_title'].nunique():,}")
print(f"Date Range: {df['trending_date'].min().strftime('%Y-%m-%d')} to {df['trending_date'].max().strftime('%Y-%m-%d')} ({df['trending_date'].nunique()} days)")

Loaded Cleaned Dataset: 32,265 rows x 29 columns
Unique Videos: 16,305
Unique Channels: 1,423
Unique Categories: 17
Date Range: 2017-11-14 to 2018-06-14 (205 days)


### 2. Analysis 1 — Trending Activity Over Time

**Business Question:** How consistently does the YouTube India trending feed refresh, and what are the daily volumes of snapshots vs active unique videos?

In [2]:
daily_activity = eda.compute_daily_trending_activity(df)
print("Daily Activity Summary (First 5 Days):")
print(daily_activity.head().to_string(index=False))

print("\nSummary Statistics of Daily Observations:")
print(f"  Mean Daily Snapshots:   {daily_activity['total_observations'].mean():.1f}")
print(f"  Median Daily Snapshots: {daily_activity['total_observations'].median():.1f}")
print(f"  Min Daily Snapshots:    {daily_activity['total_observations'].min()} on {daily_activity.loc[daily_activity['total_observations'].idxmin(), 'trending_date']}")
print(f"  Max Daily Snapshots:    {daily_activity['total_observations'].max()} on {daily_activity.loc[daily_activity['total_observations'].idxmax(), 'trending_date']}")

# Plot trending activity over time
fig_p1 = eda.plot_trending_activity(daily_activity)
print(f"Saved Figure: {fig_p1}")

Daily Activity Summary (First 5 Days):
trending_date  total_observations  unique_videos  recurring_videos
   2017-11-14                 197            197                 0
   2017-11-15                 198            198                 0
   2017-11-16                 197            197                 0
   2017-11-17                 199            199                 0
   2017-11-18                 196            196                 0

Summary Statistics of Daily Observations:
  Mean Daily Snapshots:   157.4
  Median Daily Snapshots: 161.0
  Min Daily Snapshots:    49 on 2018-05-14 00:00:00
  Max Daily Snapshots:    200 on 2017-12-13 00:00:00


Saved Figure: D:\MediaPulse\reports\figures\01_trending_activity_over_time.png


### 3. Analysis 2 — Content Category Performance & Market Share

**Business Question:** Which categories dominate video production and cumulative observed peak views, and does high content volume guarantee high audience engagement?

In [3]:
cat_summary = eda.compute_category_performance_summary(df)
print("Top 10 Categories by Unique Video Volume:")
print(cat_summary[['category_title', 'unique_videos', 'share_of_videos_pct', 'total_views', 'median_views', 'median_engagement_rate', 'mean_days_trended']].head(10).to_string(index=False))

# Verify shares sum to 100%
print(f"\nTotal Unique Videos Across Categories: {cat_summary['unique_videos'].sum():,}")
print(f"Sum of Market Shares: {cat_summary['share_of_videos_pct'].sum():.2f}%")

# Plot category performance
fig_p2 = eda.plot_category_performance(cat_summary)
print(f"Saved Figure: {fig_p2}")

Top 10 Categories by Unique Video Volume:
      category_title  unique_videos  share_of_videos_pct  total_views  median_views  median_engagement_rate  mean_days_trended
       Entertainment           7557                46.35   4349915552      202211.0                  0.5823               1.91
     News & Politics           2506                15.37    745593584      148114.5                  0.6608               1.84
      People & Blogs           1231                 7.55    553516694      177057.0                  0.6288               1.89
               Music           1214                 7.45   2449098419      555774.0                  1.9044               2.59
              Comedy           1116                 6.84    799845234      353683.0                  4.2236               2.57
           Education            797                 4.89     73816757       46273.0                  5.6205               1.44
    Film & Animation            520                 3.19    940618500

Saved Figure: D:\MediaPulse\reports\figures\02_category_performance_comparison.png


### 4. Analysis 3 — Channel-Level Publishing Strategies & Consistency

**Business Question:** Which publishers place the highest volume of unique videos on the trending list, and how do their median views and engagement rates compare?

In [4]:
top_channels = eda.compute_channel_performance_summary(df, top_n=10)
print("Top 10 Publishers by Unique Trending Videos:")
print(top_channels[['channel_title', 'unique_videos', 'total_views', 'median_views', 'mean_days_trended', 'median_engagement_rate']].to_string(index=False))

# Plot channel performance
fig_p3 = eda.plot_top_channels(top_channels)
print(f"Saved Figure: {fig_p3}")

Top 10 Publishers by Unique Trending Videos:
     channel_title  unique_videos  total_views  median_views  mean_days_trended  median_engagement_rate
Study IQ education            169      9637789       53255.0               1.18                  5.8476
         Tarang TV            158     15823246       87353.0               1.26                  0.4249
    etvteluguindia            153     49921892      212610.0               1.33                  0.3986
         VikatanTV            153    111174882      664658.0               1.31                  0.6069
  Saregama TVShows            152     23068307      157936.5               1.17                  0.6158
 Mazhavil Manorama            138     41163047      245810.0               1.38                  0.5898
       RadaanMedia            136     56058172      390547.5               1.40                  0.5420
        zeemarathi            135     18406903       96860.0               1.21                  0.3186
       zeetvtelugu 

Saved Figure: D:\MediaPulse\reports\figures\03_top_channels_analysis.png


### 5. Analysis 4 — Audience Engagement Mechanics & Power-Law Scaling

**Business Question:** How do views, likes, and comments correlate across trending content, and what does the empirical engagement rate distribution reveal?

In [5]:
corrs = eda.compute_engagement_correlations(df)
print("Engagement Metrics Correlation Matrix:")
for k, v in corrs.items():
    print(f"  {k:30s}: {v:+.4f}")

er_series = df['engagement_rate'].dropna()
print("\nEngagement Rate Distribution (%):")
print(f"  Min:    {er_series.min():.4f}%")
print(f"  P25:    {er_series.quantile(0.25):.4f}%")
print(f"  Median: {er_series.median():.4f}%")
print(f"  Mean:   {er_series.mean():.4f}%")
print(f"  P75:    {er_series.quantile(0.75):.4f}%")
print(f"  P99:    {er_series.quantile(0.99):.4f}%")
print(f"  Max:    {er_series.max():.4f}%")

# Plot scatter and distribution
fig_p4 = eda.plot_views_vs_engagement(df)
fig_p5 = eda.plot_engagement_distribution(df)
print(f"Saved Figures: {fig_p4.name}, {fig_p5.name}")

Engagement Metrics Correlation Matrix:
  pearson_views_likes           : +0.8549
  spearman_views_likes          : +0.7431
  pearson_views_comments        : +0.6994
  spearman_views_comments       : +0.6655
  pearson_likes_comments        : +0.7919
  spearman_likes_comments       : +0.8813

Engagement Rate Distribution (%):
  Min:    0.0000%
  P25:    0.4777%
  Median: 1.0527%
  Mean:   2.4232%
  P75:    3.0801%
  P99:    14.3208%
  Max:    77.4282%


Saved Figures: 04_views_vs_engagement_scatter.png, 05_engagement_rate_distribution.png


### 6. Analysis 5 — Time to Observed Trending (`days_to_trend`)

**Business Question:** How quickly do videos reach trending status after publication, and how do statistics differ between the unique-video grain and observation-level snapshots?

> **Analytical Rigor Notice on Denominators:**  
> - **Unique-Video Grain ($N = 16,305$):** Evaluates `earliest_trending_date - publish_date`. 420 videos trend on Day 0, which equals **2.58%** ($420 / 16,305$).  
> - **Observation Grain ($N = 32,265$):** Evaluates `trending_date - publish_date` across all daily snapshots. 421 rows have days_to_trend = 0, which equals **1.30%** ($421 / 32,265$).

In [6]:
vid_stats = eda.compute_video_time_to_trend_stats(df)
print("=== UNIQUE-VIDEO GRAIN (Earliest Observed Trending Date, N = 16,305) ===")
for k, v in vid_stats.items():
    print(f"  {k:25s}: {v}")

dtt_obs = df['days_to_trend']
print(f"\n=== OBSERVATION GRAIN (All Daily Snapshots, N = {len(df):,}) ===")
print(f"  Mean:             {dtt_obs.mean():.2f} days")
print(f"  Median:           {dtt_obs.median():.1f} days")
print(f"  Day 0 Rows:       {(dtt_obs == 0).sum():,} ({(dtt_obs == 0).sum() / len(dtt_obs) * 100:.2f}% of all snapshot rows)")
print(f"  Within 2 Days:    {(dtt_obs <= 2).sum():,} ({(dtt_obs <= 2).sum() / len(dtt_obs) * 100:.2f}% of all snapshot rows)")
print(f"  Within 7 Days:    {(dtt_obs <= 7).sum():,} ({(dtt_obs <= 7).sum() / len(dtt_obs) * 100:.2f}% of all snapshot rows)")

time_summary = eda.compute_time_to_trend_summary(df)
print("\nTime to First Observed Trending by Category (Unique Video Grain):")
print(time_summary.head(8).to_string(index=False))

# Plot time to trend by category
fig_p6 = eda.plot_time_to_trend(df)
print(f"Saved Figure: {fig_p6}")

=== UNIQUE-VIDEO GRAIN (Earliest Observed Trending Date, N = 16,305) ===
  total_unique_videos      : 16305
  mean_days                : 1.31
  median_days              : 1.0
  p25_days                 : 1.0
  p75_days                 : 2.0
  min_days                 : 0
  max_days                 : 220
  day_0_count              : 420
  day_0_pct                : 2.58
  within_2_days_count      : 15720
  within_2_days_pct        : 96.41
  within_7_days_count      : 16285
  within_7_days_pct        : 99.88

=== OBSERVATION GRAIN (All Daily Snapshots, N = 32,265) ===
  Mean:             2.11 days
  Median:           2.0 days
  Day 0 Rows:       421 (1.30% of all snapshot rows)
  Within 2 Days:    22,731 (70.45% of all snapshot rows)
  Within 7 Days:    32,201 (99.80% of all snapshot rows)



Time to First Observed Trending by Category (Unique Video Grain):
  category_title  median_days_to_trend  mean_days_to_trend  p25_days  p75_days  min_days  max_days
Autos & Vehicles                   1.0                1.18       1.0       1.0         0         2
          Comedy                   1.0                1.20       1.0       1.0         0         5
       Education                   1.0                0.96       1.0       1.0         0         3
   Entertainment                   1.0                1.34       1.0       2.0         0       220
Film & Animation                   1.0                1.44       1.0       2.0         0        49
          Gaming                   1.0                1.30       1.0       1.0         0         5
   Howto & Style                   1.0                1.31       1.0       2.0         0         4
          Movies                   1.0                1.00       0.5       1.5         0         2


Saved Figure: D:\MediaPulse\reports\figures\06_time_to_trend_by_category.png


### 7. Analysis 6 — Trending Persistence Across Discrete Snapshot Dates

**Business Question:** Across how many distinct daily snapshot dates do unique videos appear on the trending list?

> **Analytical Rigor Notice on Presence:** The dataset captures discrete daily scrape snapshots. It does not establish continuous, uninterrupted 24-hour presence or observe intraday additions/removals. Observations represent distinct calendar dates on which a video was captured in the trending feed.

In [7]:
persistence_df = eda.compute_trending_persistence_summary(df)
print("Trending Persistence (Distinct Snapshot Dates per Unique Video):")
print(persistence_df.to_string(index=False))

# Plot persistence distribution
fig_p7 = eda.plot_trending_persistence(df)
print(f"Saved Figure: {fig_p7}")

Trending Persistence (Distinct Snapshot Dates per Unique Video):
persistence_bracket  video_count  percentage
              1 Day         7824       47.99
             2 Days         4168       25.56
           3-5 Days         4087       25.07
          6-10 Days          226        1.39
           11+ Days            0        0.00


Saved Figure: D:\MediaPulse\reports\figures\07_trending_persistence_distribution.png


### 8. Analysis 7 — Publication Timing Patterns

**Business Question:** When are trending videos most frequently published, and how should publication hour data be interpreted?

> **Analytical Rigor Notice:** Publish hours are stored in UTC. In Indian Standard Time (IST), add 5 hours 30 minutes (e.g. 14:00 UTC = 19:30 IST). Furthermore, correlation does not imply causation: popular creators schedule uploads during evening viewing windows, creating volume concentration.

In [8]:
weekday_df, hour_df = eda.compute_publication_patterns(df)
print("Publication Volume by Weekday:")
print(weekday_df.to_string(index=False))

print("\nTop 5 Publication Hours (UTC):")
print(hour_df.sort_values(by='unique_videos', ascending=False).head(5).to_string(index=False))

# Plot publication patterns
fig_p8 = eda.plot_publication_patterns(weekday_df, hour_df)
print(f"Saved Figure: {fig_p8}")

Publication Volume by Weekday:
  weekday  unique_videos
   Monday           2246
  Tuesday           2391
Wednesday           2332
 Thursday           2503
   Friday           2694
 Saturday           2421
   Sunday           1720

Top 5 Publication Hours (UTC):
 publish_hour_utc  unique_videos
               14           1117
               16           1031
               12           1027
               13            992
               15            986


Saved Figure: D:\MediaPulse\reports\figures\08_publication_patterns.png


### 9. Summary of Key Analytical Findings for Business Intelligence

1. **Volume Concentration:** Entertainment and News account for **61.72%** of all unique trending videos and **5.09B** cumulative observed peak views.
2. **Engagement Asymmetry:** Educational and Tech content achieves **5x–12x higher median engagement rates** than mainstream entertainment.
3. **Monotonic Engagement Scaling:** Views and likes exhibit strong rank alignment (Spearman $\rho = +0.7431$, Pearson $r = +0.8549$).
4. **Rapid Viral Onset:** **96.41%** of unique videos appear on the trending leaderboard within $\le 2$ days of publication (median = 1.0 day). Day-0 breakout videos account for **2.58%** of unique videos ($420 / 16,305$).
5. **Discrete Snapshot Decay:** **47.99%** of videos appear on only a single trending snapshot date, and **73.55%** appear on $\le 2$ distinct dates.
6. **Publishing Cadence:** Peak publication occurs on Friday (2,694 videos) and Thursday (2,503 videos), concentrated in the 12:00–16:00 UTC window (17:30–21:30 IST prime time).